```
  set Density (rho)                           = 1
  set Elastic modulus of the vessel wall (mu) = 1
  set Reference cross-sectional area (a0)     = 1
  set Reference pressure (p0)                 = 1
  set Reference radius (r0)                   = 1
  set Tube law exponent (m)                   = 0.5
  set Viscosity coefficient (eta_c)           = 1
```

In [1]:
import sympy
from sympy import *
rho, mu, a0, p0, r0, m, eta_c, x, t = symbols('rho mu a_0 p_0 r_0 m eta_c x t')

A = Function('A')(x, t)
U = Function('U')(x, t)

dA = Function('\delta A')(x, t)
dU = Function('\delta U')(x, t)

fA = Function('f_A')(x, t)
fU = Function('f_U')(x, t)

# pressure
P = p0 + mu*( (A/a0)**m - 1 )

# fluxes
flux_A = A*U
flux_U = U**2/2 + P/rho

# manufactured RHS
fA_man = simplify(diff(A, t) + diff(flux_A, x) )
fU_man = simplify(diff(U, t) + diff(flux_U, x) + eta_c * U)

In [2]:
# implicit function for ARKODE
LA = fA - diff(flux_A, x)
LU = fU - diff(flux_U, x) - eta_c * U

L = Matrix([LA, LU])
W = Matrix([A, U])
dW = Matrix([dA, dU])

# Frechet derivative of L in the direction dW (symbolic expression of J*dW)
eps = symbols('epsilon')
JdW = (L.subs({A: A + eps*dA, U: U + eps * dU}) - L).diff(eps).subs({eps: 0})

In [3]:
Aexp = a0*(1+.5*sin(2*pi*x-t*pi))
Aexp = x+1
Aexp = 0*x+1
Uexp = 0*x

dAexp = 0*x+1
dUexp = 0*x+1

A0exp = Aexp.subs({t:0})
U0exp = Uexp.subs({t:0})

s = {A: Aexp, U: Uexp,
     dA: dAexp, dU: dUexp}

fA_man_exp = fA_man.subs(s).simplify()
fU_man_exp = fU_man.subs(s).simplify()

Lexp = L.subs(s).subs({fA:fA_man_exp, fU:fU_man_exp})

JdWexp = JdW.subs(s)

In [4]:
s = f"""
   set Exact solution = {Aexp}; {Uexp}
   set Initial condition = {A0exp}; {U0exp}
   set RHS expression = {fA_man_exp}; {fU_man_exp}

   set L(W) = {Lexp[0].simplify()}; {Lexp[1].simplify()}
   set dW = {dAexp}; {dUexp}
   set J*dW = {JdWexp[0,0].simplify()}; {JdWexp[1,0].simplify()}
"""

print(s.replace('pi', 'PI').replace('**', '^'))


   set Exact solution = 1; 0
   set Initial condition = 1; 0
   set RHS expression = 0; 0

   set L(W) = 0; 0
   set dW = 1; 1
   set J*dW = 0; -eta_c

